# 03 — Self-Attention Mechanism

> **Goal:** Understand the single most important formula in modern AI.

**The formula:** `Attention(Q, K, V) = softmax(Q @ K.T / √d_k) @ V`

This one formula is the heart of GPT, Claude, BERT, and every modern LLM.

---

### Why do we need Attention?

Consider this sentence:
```
"The animal didn't cross the street because it was too tired"
```
What does **"it"** refer to? → **"animal"**, not "street".

Your brain figures this out by paying **attention** to the right words.
The attention mechanism does exactly the same thing.


In [ ]:
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print('Libraries loaded ✓')

Libraries loaded ✓


---
## Step 1 — Input (Embeddings)

Attention takes word embeddings as input — the output from our NLP notebook.

Each row = one word, each column = one feature.


In [ ]:
# ── Step 1: Input Embeddings ──────────────────────────────
# Sentence: "I love AI"
# Each word → 4-dimensional embedding vector

words = ['I', 'love', 'AI']

X = np.array([
    [1.0, 0.0, 1.0, 0.0],   # 'I'
    [0.0, 1.0, 0.0, 1.0],   # 'love'
    [1.0, 1.0, 0.0, 0.0],   # 'AI'
])

print('Input: "I love AI"')
print(f'Shape: {X.shape}  ← ({len(words)} words × 4 features)')
print()
for i, w in enumerate(words):
    print(f"  '{w}' → {X[i]}")

Input: "I love AI"
Shape: (3, 4)  ← (3 words × 4 features)

  'I' → [1. 0. 1. 0.]
  'love' → [0. 1. 0. 1.]
  'AI' → [1. 1. 0. 0.]


---
## Step 2 — Weight Matrices W_Q, W_K, W_V

Three weight matrices that are **learned during training**:

| Matrix | Meaning | Question it answers |
|--------|---------|--------------------|
| **W_Q** | Query weights | What am I looking for? |
| **W_K** | Key weights | What information do I have? |
| **W_V** | Value weights | What do I actually pass along? |

Think of a **library**:
- Query = the question you ask the librarian
- Key = the label on each book shelf
- Value = the actual content of the book

In [ ]:
# ── Step 2: Weight Matrices ───────────────────────────────
d_model = 4   # input embedding size
d_k     = 3   # size of Q, K, V

# Random at start — learned during training
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)
W_V = np.random.randn(d_model, d_k)

print('Weight matrices (random at start, learned during training):')
print(f'  W_Q shape: {W_Q.shape}  ← (embedding=4) → (query_dim=3)')
print(f'  W_K shape: {W_K.shape}  ← (embedding=4) → (key_dim=3)')
print(f'  W_V shape: {W_V.shape}  ← (embedding=4) → (value_dim=3)')

Weight matrices (random at start, learned during training):
  W_Q shape: (4, 3)  ← (embedding=4) → (query_dim=3)
  W_K shape: (4, 3)  ← (embedding=4) → (key_dim=3)
  W_V shape: (4, 3)  ← (embedding=4) → (value_dim=3)


---
## Step 3 — Compute Q, K, V

Simple matrix multiplication:
```
Q = X @ W_Q   ← "What is each word asking for?"
K = X @ W_K   ← "What does each word offer?"
V = X @ W_V   ← "What is the actual content?"
```

**فارسی:** فقط ضرب ماتریسی ساده. Q = سوال هر کلمه، K = اطلاعات هر کلمه، V = محتوای واقعی.

In [ ]:
# ── Step 3: Compute Q, K, V ───────────────────────────────
Q = X @ W_Q   # shape: (3, 3)
K = X @ W_K   # shape: (3, 3)
V = X @ W_V   # shape: (3, 3)

print('Q — What is each word asking for?')
for i, w in enumerate(words):
    print(f"  '{w}' asks: {Q[i]}")

print('\nK — What does each word offer?')
for i, w in enumerate(words):
    print(f"  '{w}' offers: {K[i]}")

print('\nV — Actual content of each word:')
for i, w in enumerate(words):
    print(f"  '{w}' content: {V[i]}")

Q — What is each word asking for?
  'I' asks: [ 0.093 -2.261 -2.807]
  'love' asks: [-0.523  0.278  1.228]
  'AI' asks: [ 0.406 -1.221 -1.157]

K — What does each word offer?
  'I' offers: [ 1.375 -0.832 -0.515]
  'love' offers: [-0.694 -0.346  1.587]
  'AI' offers: [-0.041 -2.44   0.936]

V — Actual content of each word:
  'I' content: [-0.551  0.818 -0.745]
  'love' content: [-1.841  1.174  2.894]
  'AI' content: [-1.675  0.627  0.25 ]


---
## Step 4 — Attention Scores

For every pair of words, compute: **how related are they?**

```
scores = Q @ K.T / √d_k
```

- `Q @ K.T` = dot product = similarity between query and key
- `/ √d_k` = scale down to prevent values from getting too large


In [ ]:
# ── Step 4: Attention Scores ──────────────────────────────
scores_raw   = Q @ K.T                    # raw dot products
scores_scaled = scores_raw / np.sqrt(d_k) # scaled

print('Attention Scores = Q @ K.T / √d_k')
print(f'(higher score = more related)')
print()
print(f"{'':10}", end='')
for w in words:
    print(f"  K['{w}'] ", end='')
print()
for i, wi in enumerate(words):
    print(f"  Q['{wi}']{'':<3}", end='')
    for j in range(len(words)):
        print(f"  {scores_scaled[i][j]:7.3f} ", end='')
    print()

print(f'\n√d_k = √{d_k} = {np.sqrt(d_k):.3f}  ← we divide by this')

Attention Scores = Q @ K.T / √d_k
(higher score = more related)

            K['I']   K['love']   K['AI'] 
  Q['I']       1.994    -2.158     1.666 
  Q['love']      -0.914     1.280     0.285 
  Q['AI']       1.252    -0.979     1.086 

√d_k = √3 = 1.732  ← we divide by this


---
## Step 5 — Softmax → Attention Weights

Convert raw scores into **probabilities** that sum to 1.0:

```
scores = [-2.03, -1.40, -1.59]    ← raw numbers
weights = [0.23,  0.42,  0.35]    ← probabilities (sum = 1.0)
```

This tells us: "pay 23% attention to word 1, 42% to word 2, 35% to word 3"

In [ ]:
# ── Step 5: Softmax → Attention Weights ──────────────────

def softmax(x):
    """Convert scores to probabilities. Each row sums to 1.0"""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))  # numerical stability
    return e_x / e_x.sum(axis=-1, keepdims=True)

attention_weights = softmax(scores_scaled)

print('Attention Weights (each row sums to 1.0):')
print()
print(f"{'':10}", end='')
for w in words:
    print(f"  to '{w}'  ", end='')
print()
for i, wi in enumerate(words):
    print(f"  '{wi}'{'':<5}", end='')
    for j in range(len(words)):
        w = attention_weights[i][j]
        bar = '█' * int(w * 15)
        print(f"  {w:.2f} {bar:<10}", end='')
    print()

print('\nInterpretation:')
for i, wi in enumerate(words):
    j = np.argmax(attention_weights[i])
    pct = attention_weights[i][j]
    print(f"  '{wi}' pays most attention to '{words[j]}': {pct:.0%}")

# Verify rows sum to 1
print(f'\nRow sums: {attention_weights.sum(axis=1)}  ← all 1.0 ✓')

Attention Weights (each row sums to 1.0):

            to 'I'    to 'love'    to 'AI'  
  'I'       0.58 ████████    0.01             0.41 ██████    
  'love'       0.08 █           0.68 ██████████  0.25 ███       
  'AI'       0.51 ███████     0.05             0.43 ██████    

Interpretation:
  'I' pays most attention to 'I': 58%
  'love' pays most attention to 'love': 68%
  'AI' pays most attention to 'I': 51%

Row sums: [1. 1. 1.]  ← all 1.0 ✓


---
## Step 6 — Output

Final step: weighted combination of Values.

```
output = attention_weights @ V
```

For word 'love' with weights [0.28, 0.13, 0.58]:
```
output['love'] = 0.28 × V['I'] + 0.13 × V['love'] + 0.58 × V['AI']
```

Now 'love' contains information from ALL words, weighted by attention!


In [ ]:
# ── Step 6: Output ────────────────────────────────────────
output = attention_weights @ V   # shape: (3, 3)

print('Output = attention_weights @ V')
print()

# Show the calculation for 'love'
i = 1  # 'love'
print(f"Detailed calculation for '{words[i]}'")
print(f"  weights: {attention_weights[i]}")
total = np.zeros(d_k)
for j, wj in enumerate(words):
    contrib = attention_weights[i][j] * V[j]
    total += contrib
    print(f"  {attention_weights[i][j]:.2f} × V['{wj}'] = {contrib}")
print(f"  sum = {total}")
print(f"  output['{words[i]}'] = {output[i]}  ✓")

print('\nAll outputs:')
for i, w in enumerate(words):
    print(f"  '{w}':")
    print(f"    before attention: {X[i]}")
    print(f"    after attention:  {output[i]}")
    print(f"    → now contains context from other words!")

Output = attention_weights @ V

Detailed calculation for 'love'
  weights: [0.075 0.675 0.25 ]
  0.08 × V['I'] = [-0.041  0.062 -0.056]
  0.68 × V['love'] = [-1.243  0.793  1.954]
  0.25 × V['AI'] = [-0.418  0.156  0.062]
  sum = [-1.703  1.011  1.96 ]
  output['love'] = [-1.703  1.011  1.96 ]  ✓

All outputs:
  'I':
    before attention: [1. 0. 1. 0.]
    after attention:  [-1.029  0.742 -0.299]
    → now contains context from other words!
  'love':
    before attention: [0. 1. 0. 1.]
    after attention:  [-1.703  1.011  1.96 ]
    → now contains context from other words!
  'AI':
    before attention: [1. 1. 0. 0.]
    after attention:  [-1.109  0.755 -0.114]
    → now contains context from other words!


In [ ]:
## Complete Attention Function
# ── Complete Attention in One Function ───────────────────

def self_attention(X, W_Q, W_K, W_V):
    """
    Self-Attention mechanism.

    Args:
        X:   input embeddings  shape: (seq_len, d_model)
        W_Q: query weights     shape: (d_model, d_k)
        W_K: key weights       shape: (d_model, d_k)
        W_V: value weights     shape: (d_model, d_k)

    Returns:
        output: context-aware representations  shape: (seq_len, d_k)
    """
    d_k = W_K.shape[1]

    # Step 1: compute Q, K, V
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    # Step 2: attention scores
    scores = Q @ K.T / np.sqrt(d_k)

    # Step 3: softmax
    e = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights = e / e.sum(axis=-1, keepdims=True)

    # Step 4: weighted sum of values
    return weights @ V


# Test
result = self_attention(X, W_Q, W_K, W_V)
print('self_attention(X, W_Q, W_K, W_V):')
print(f'Input shape:  {X.shape}')
print(f'Output shape: {result.shape}')
print()
print('The formula in one line:')
print('output = softmax( Q @ K.T / √d_k ) @ V')
print()
print('This is the heart of GPT, Claude, and every modern LLM. ✓')

self_attention(X, W_Q, W_K, W_V):
Input shape:  (3, 4)
Output shape: (3, 3)

The formula in one line:
output = softmax( Q @ K.T / √d_k ) @ V

This is the heart of GPT, Claude, and every modern LLM. ✓


---
## Summary
**6 Steps of Attention:**

| Step | Operation | Meaning |
|------|-----------|--------|
| 1 | `X` | Input embeddings |
| 2 | `W_Q, W_K, W_V` | Learned weight matrices |
| 3 | `Q=X@W_Q, K=X@W_K, V=X@W_V` | Query, Key, Value |
| 4 | `scores = Q @ K.T / √d_k` | How related are words? |
| 5 | `weights = softmax(scores)` | Convert to probabilities |
| 6 | `output = weights @ V` | Weighted combination |

**The formula:**
```
Attention(Q, K, V) = softmax( Q @ K.T / √d_k ) @ V
```
**Next:** `04_mini_gpt.ipynb` — stack attention + feedforward to build a real language model